# 02 · Bronze → Silver: Limpeza, Tipagem e Padronização

A primeira vista, é possível notar algumas inconsistências nos dados, como:
- **Nos atendentes:** tem nível misturando `'1'`, `'Nível 1'`, `'2'`, `'N2'`; datas em formato de texto (`dd/MM/yyyy`); e campos vazios quando a pessoa ainda não foi desligada.
- **Nos motivos:** tem categorias que vieram em branco (`""`), algumas escritas sem acento (`Cartao`, `Beneficios`), e criticidade toda misturada (`alta`, `Media`, `baixa`).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Lendo da Bronze: {bronze_schema}")
print(f"Gravando na Silver: {silver_schema}")


---
## 1. Tratamento dos Atendentes (`silver.tb_atendentes`)

O que está sendo feito:
- **`id_atendente`**: passa de string pra número inteiro (`IntegerType`);
- **`nivel_atendimento`**: unifica tudo em **`Nível 1`** ou **`Nível 2`** (troca '1' e 'N1' por 'Nível 1', e '2' e 'N2' por 'Nível 2');
- **`data_admissao`**: converte o texto brasileiro `dd/MM/yyyy` pra data de verdade (`DateType`);
- **`data_desligamento`**: quem tá com string vazia vira `NULL` (quem ainda tá ativo na empresa), e quem tem data vira `DateType`;
- **`status_atendente`**: padroniza em caixa alta inicial (`Ativo` / `Inativo`).


In [0]:
df_bronze_atendentes = spark.table(f"{bronze_schema}.tb_atendentes")

win_atendente = Window.partitionBy("id_atendente").orderBy(F.col("ingestion_datetime").desc())

df_silver_atendentes = (
    df_bronze_atendentes
    .withColumn("rn", F.row_number().over(win_atendente))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("id_atendente").cast(IntegerType()).alias("id_atendente"),
        
        F.trim(F.col("nome_atendente")).alias("nome_atendente"),
        
        F.when(F.col("nivel_atendimento").isin("1", "Nível 1", "Nivel 1", "N1"), "Nível 1")
         .when(F.col("nivel_atendimento").isin("2", "Nível 2", "Nivel 2", "N2"), "Nível 2")
         .otherwise("Não Informado")
         .alias("nivel_atendimento"),
        
        F.to_date(F.col("data_admissao"), "dd/MM/yyyy").alias("data_admissao"),
        
        F.when(
            (F.col("data_desligamento").isNull()) | (F.trim(F.col("data_desligamento")) == ""), 
            F.lit(None)
        ).otherwise(
            F.to_date(F.col("data_desligamento"), "dd/MM/yyyy")
        ).alias("data_desligamento"),
        
        F.initcap(F.trim(F.col("status_atendente"))).alias("status_atendente"),
        
        F.col("ingestion_datetime")
    )
)

(
    df_silver_atendentes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_atendentes")
)

print(f"Tabela {silver_schema}.tb_atendentes salva com sucesso!")
print(f"Total de atendentes: {df_silver_atendentes.count()}")

In [0]:
print("Contagem por Nível e Status:")
df_silver_atendentes.groupBy("nivel_atendimento", "status_atendente").count().show()

display(df_silver_atendentes.limit(5))

---
## 2. Tratamento dos Motivos (`silver.dim_motivo`)

O que está sendo feito:
- **`id_motivo`**: vira número inteiro (`IntegerType`);
- **Preenchimento de categorias em branco**: no arquivo original vieram 3 motivos sem categoria. Esses foram os critérios usados para classificação:
  - *'Transferência de agência'* $\rightarrow$ **Cadastral** (dados cadastrais do cliente/agência);
  - *'Contratação de cartão adicional'* $\rightarrow$ **Cartão** (operações e contratações do cartão);
  - *'Cancelamento de conta'* $\rightarrow$ **Financeiro** (contrato/encerramento da conta com o banco);
- **Acentuação**: arruma `'Cartao'` pra `'Cartão'` e `'Beneficios'` pra `'Benefícios'`;
- **Criticidade**: padroniza palavras com mistura de maiúsculas/minúsculas pra **`Alta`**, **`Média`** ou **`Baixa`**.

In [0]:
df_bronze_motivos = spark.table(f"{bronze_schema}.tb_motivos")

win_motivo = Window.partitionBy("id_motivo").orderBy(F.col("ingestion_datetime").desc())

df_silver_motivos = (
    df_bronze_motivos
    .withColumn("rn", F.row_number().over(win_motivo))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("id_motivo").cast(IntegerType()).alias("id_motivo"),
        
        F.trim(F.col("nome_motivo")).alias("nome_motivo"),
        
        F.when(F.col("nome_motivo") == "Transferência de agência", "Cadastral")
         .when(F.col("nome_motivo") == "Contratação de cartão adicional", "Cartão")
         .when(F.col("nome_motivo") == "Cancelamento de conta", "Financeiro")
         .when(F.lower(F.col("categoria")).isin("cartao", "cartão"), "Cartão")
         .when(F.lower(F.col("categoria")).isin("beneficios", "benefícios"), "Benefícios")
         .when(F.lower(F.col("categoria")) == "financeiro", "Financeiro")
         .when(F.lower(F.col("categoria")) == "cadastral", "Cadastral")
         .when(F.lower(F.col("categoria")) == "atendimento", "Atendimento")
         .otherwise(F.initcap(F.trim(F.col("categoria"))))
         .alias("categoria"),
        
        F.when(F.lower(F.col("criticidade")).isin("alta"), "Alta")
         .when(F.lower(F.col("criticidade")).isin("media", "média"), "Média")
         .when(F.lower(F.col("criticidade")).isin("baixa"), "Baixa")
         .otherwise(F.initcap(F.trim(F.col("criticidade"))))
         .alias("criticidade"),
        
        F.col("ingestion_datetime")
    )
)

(
    df_silver_motivos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_motivos")
)

print(f"Tabela {silver_schema}.tb_motivos salva com sucesso!")
print(f"Total de motivos: {df_silver_motivos.count()}")


In [0]:
print("Resumo das categorias e criticidades tratadas:")
df_silver_motivos.groupBy("categoria", "criticidade").count().orderBy("categoria").show()

display(df_silver_motivos.orderBy("id_motivo"))


Assim, as tabelas `silver.tb_atendentes` e `silver.tb_motivos` estão prontas, higienizadas e padronizadas.
Todos os valores em branco das categorias foram tratados, os tipos estão ajustados corretamente e o pipeline pode rodar livremente sem travamento ou duplicação de dados.
